# Jaxotron tour: 4D parallelism in JAX 🚀

This notebook walks through **data, FSDP, tensor, pipeline and context parallelism** with
[Jaxotron](https://github.com/rishiraj/jaxotron). Every command runs in two places:

* **Colab / any CPU machine**: `--cpu_devices N` simulates N accelerators, so all layouts work without a GPU.
* **Kaggle TPU v5e-8 (free)**: set the accelerator to *TPU VM v5e-8* and use the TPU section at the end.

Each training command runs in its own process, so simulated devices are configured before JAX starts.

In [ ]:
!pip install -q "jaxotron[hf,data] @ git+https://github.com/rishiraj/jaxotron"
!python -c "import jax; print('JAX', jax.__version__, '|', jax.devices())"

## 1. Baseline: one device

A learnable synthetic dataset (arithmetic progressions) is generated automatically, so the loss visibly drops in a few dozen steps.

In [ ]:
SMALL = "--seq_len 128 --global_batch_size 16 --micro_batch_size 4 --log_every 10"
!jaxotron-train --cpu_devices 1 {SMALL} --steps 30 --checkpoint_dir /tmp/tour/1dev

## 2. FSDP (ZeRO-3): every weight split 8 ways

`--show_sharding` prints the `PartitionSpec` of every parameter and how many MB each device holds.
The rule that does this is a single line in `jaxotron/parallel/rules.py`: `"embed": "fsdp"`.

In [ ]:
!jaxotron-train --cpu_devices 8 --fsdp 8 {SMALL} --micro_batch_size 2 --dry_run --show_sharding --checkpoint_dir /tmp/tour/fsdp

## 3. Tensor + context parallelism, and the communication XLA inserted

We never write an `all_reduce`. GSPMD adds the collectives, and `--show_collectives` reads them back from the compiled program:
tensor parallelism appears as all-reduces over `tp`, context parallelism as an all-gather of keys/values over `cp`.

In [ ]:
!jaxotron-train --cpu_devices 4 --tp 2 --cp 2 {SMALL} --steps 30 --show_collectives --checkpoint_dir /tmp/tour/tpcp

## 4. Full 4D: DP × PP × CP × TP on 16 devices

`--show_mesh` prints which `(dp, pp, cp, tp)` coordinates every device owns. The pipeline bubble is reported at startup.

In [ ]:
!jaxotron-train --cpu_devices 16 --dp 2 --pp 2 --cp 2 --tp 2 {SMALL} --steps 40 --show_mesh --show_collectives --checkpoint_dir /tmp/tour/4d

## 5. Topology-independent checkpoints

The checkpoint above was written by 16 devices. Orbax reshards it onto a single device for sampling,
and training could equally resume on any other layout. The model continues the arithmetic progression.

In [ ]:
!jaxotron-generate --cpu --checkpoint_dir /tmp/tour/4d --prompt "5 8 11 14" --max_new_tokens 16 --temperature 0

## 6. A real pretrained model: SmolLM2-135M with tensor parallelism

Hugging Face safetensors load straight into the sharded layout (no PyTorch). SmolLM2-135M has 9 attention heads
and 3 KV heads, so it splits cleanly over 3 tensor-parallel devices.

In [ ]:
!jaxotron-generate --hf HuggingFaceTB/SmolLM2-135M --cpu_devices 3 --tp 3 --prompt "The three laws of motion are" --max_new_tokens 40 --temperature 0

## 7. Kaggle TPU v5e-8: continued pre-training of SmolLM2-135M

Switch the notebook accelerator to **TPU VM v5e-8**, restart, re-run the install cell, then run the cells below.
These use the 8 real TPU chips (no `--cpu_devices`). Training uses bf16 automatically on TPU.

In [ ]:
!jaxotron-prepare-data --dataset roneneldan/TinyStories --tokenizer HuggingFaceTB/SmolLM2-135M --out_dir data/tinystories --max_docs 100000

In [ ]:
!jaxotron-train --init_from_hf HuggingFaceTB/SmolLM2-135M --dp 2 --fsdp 4 \
    --data data/tinystories/train.bin --seq_len 1024 --global_batch_size 64 --micro_batch_size 8 \
    --lr 3e-4 --warmup 50 --steps 300 --eval_every 100 --save_every 100 --log_every 20 \
    --show_collectives --checkpoint_dir checkpoints/smollm2-tinystories

In [ ]:
!jaxotron-generate --checkpoint_dir checkpoints/smollm2-tinystories --prompt "Once upon a time" --max_new_tokens 80

## Where to go next

* Open `jaxotron/parallel/rules.py`, change a rule (for example `"mlp": None`), and re-run step 3 to watch the collectives change.
* Read `jaxotron/parallel/pipeline.py`: the whole pipeline schedule is ~40 lines, and `jax.grad` writes the backward pass.
* ⭐ the repo and share what you build: https://github.com/rishiraj/jaxotron